# Stochastic Calculus from Zero · Episode 3: Maths pit stop: averages settle down

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Coin tosses are simulated with a fixed seed: heads has a 60% chance, as in the coin world.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The running average · 4. Variance · 5. The bell appears · 6. One over root n · 7. This episode's question · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 3,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

Exactly two heads: HHT, HTH and THH, each with weight $0.6^2 \times 0.4 = 0.144$, so $3 \times 0.144 = 0.432$.

In [ ]:
p = 0.6
out["prev"] = {"answer": 3 * p * p * (1 - p)}
assert abs(out["prev"]["answer"] - 0.432) < 1e-12

## 3. The running average

Toss the 60% coin again and again and track the fraction of heads so far. Early on it jumps around; as tosses pile up it settles on 0.6. That is the **law of large numbers**: the average of many independent tosses settles on the expectation.

In [ ]:
rng = np.random.default_rng(seed)
tosses = rng.random(10_000) < p
running = np.cumsum(tosses) / np.arange(1, tosses.size + 1)
marks = (10, 100, 1_000, 10_000)
# chart on a log scale of tosses: x = log10(n)
grid = np.unique(np.round(np.logspace(0, 4, 200)).astype(int))
out["running"] = {"fraction": {str(n): float(running[n - 1]) for n in marks},
                  "x": np.log10(grid).round(4).tolist(), "y": running[grid - 1].round(4).tolist(),
                  "target": [p] * len(grid),
                  "ticks": [{"x": i, "label": f"{10 ** i:,}"} for i in range(5)]}
assert abs(running[-1] - p) < 0.02
plt.figure(figsize=(7, 3)); plt.semilogx(grid, running[grid - 1]); plt.axhline(p, ls="--"); plt.title("Fraction of heads so far"); plt.show()
out["running"]["fraction"]

## 4. Variance

One toss gives heads (1) or tails (0). Its mean is $p = 0.6$ and its variance is $p(1-p) = 0.24$. Independent tosses add their variances, so 100 tosses have variance 24 and standard deviation $\sqrt{24} \approx 4.90$.

In [ ]:
var1 = p * (1 - p)
out["variance"] = {"one": var1, "n": 100, "hundred": 100 * var1, "sd_hundred": math.sqrt(100 * var1), "mean_hundred": 100 * p}
out["variance"]

## 5. The bell appears

Repeat the 100 tosses 10,000 times and count the heads each time. The counts pile up in a bell shape around 60 (the **central limit theorem**). About two-thirds land within one standard deviation and about 95% within two. Heads come in whole numbers, so "within 4.90 of 60" really means 56 to 64, a little under the smooth curve's 68.3%.

In [ ]:
runs = 10_000
heads = (rng.random((runs, 100)) < p).sum(axis=1)
sd = out["variance"]["sd_hundred"]
k = np.arange(35, 86)
share = np.array([(heads == x).mean() for x in k])
curve_x = np.linspace(35, 85, 201)
curve_y = np.exp(-0.5 * ((curve_x - 60) / sd) ** 2) / (sd * math.sqrt(2 * math.pi))
out["bell"] = {"runs": runs, "mean": float(heads.mean()), "sd": float(heads.std()),
               "within_1sd": float((np.abs(heads - 60) <= sd).mean()), "within_2sd": float((np.abs(heads - 60) <= 2 * sd).mean()),
               "k": k.tolist(), "share": share.round(5).tolist(), "curve_x": curve_x.round(3).tolist(), "curve_y": curve_y.round(5).tolist(),
               "lo": 60 - sd, "hi": 60 + sd, "x_range": [35, 85],
               "ticks": [{"x": x, "label": str(x)} for x in range(40, 85, 10)]}
assert abs(out["bell"]["sd"] / sd - 1) < 0.03
assert 0.6 < out["bell"]["within_1sd"] < 0.7 and 0.93 < out["bell"]["within_2sd"] < 0.97
{k_: v for k_, v in out["bell"].items() if k_ in ("mean", "sd", "within_1sd", "within_2sd")}

## 6. One over root n

The *fraction* of heads in $n$ tosses has standard deviation $\sqrt{p(1-p)/n}$: four times the tosses halves the wobble. Monte Carlo pricing (Series 3, Episode 17) inherits exactly this: its error shrinks like $1/\sqrt{n}$.

In [ ]:
wobble = []
for n in (100, 400, 1_600, 6_400):
    frac = (rng.random((4_000, n)) < p).mean(axis=1)
    wobble.append({"tosses": n, "sd_sim": float(frac.std()), "sd_theory": math.sqrt(var1 / n)})
    assert abs(wobble[-1]["sd_sim"] / wobble[-1]["sd_theory"] - 1) < 0.05
out["wobble"] = {"rows": wobble, "sd_100": wobble[0]["sd_theory"], "sd_400": wobble[1]["sd_theory"]}
pd.DataFrame(wobble)

## 7. This episode's question

How many tosses halve the spread of the heads fraction you get from 100? The spread goes like $1/\sqrt{n}$, so you need four times as many: 400.

In [ ]:
out["question"] = {"from": 100, "answer": 400, "ratio": out["wobble"]["sd_100"] / out["wobble"]["sd_400"]}
assert abs(out["question"]["ratio"] - 2) < 1e-12

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** In the coin world, $S_1$ is 120 (chance 0.6) or 80 (chance 0.4). Find its mean, variance and standard deviation.

<details><summary>Answer</summary>

Mean $0.6 \times 120 + 0.4 \times 80 = 104$. Variance $0.6 \times 16^2 + 0.4 \times 24^2 = 153.6 + 230.4 = 384$. Standard deviation $\sqrt{384} \approx 19.6$.

</details>

**Exercise 2 (By hand).** What is the standard deviation of the number of heads in 400 tosses of the 60% coin?

<details><summary>Answer</summary>

Variance $400 \times 0.24 = 96$, so the standard deviation is $\sqrt{96} \approx 9.80$.

</details>

**Exercise 3 (By hand).** After five tails in a row, what is the chance the next toss is heads?

<details><summary>Answer</summary>

Still 0.6: the tosses are independent. The law of large numbers works by dilution (a run of tails becomes a small part of a long history), not by the coin making up for it.

</details>

**Exercise 4 (In code).** Using Section 6's table, fit a line to log(spread) against log(tosses). What slope do you get?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
x = np.log([r["tosses"] for r in wobble]); y = np.log([r["sd_sim"] for r in wobble])
slope = np.polyfit(x, y, 1)[0]
assert -0.55 < slope < -0.45
print(f"slope {slope:.3f}: the wobble shrinks like 1/sqrt(n)")

**Exercise 5 (In code).** Average $S_1$ over 50 simulated tosses, and repeat 5,000 times. Is the spread of those averages close to $19.6/\sqrt{50}$?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
s1 = np.where(rng.random((5_000, 50)) < p, 120.0, 80.0).mean(axis=1)
assert abs(s1.mean() - 104) < 0.2 and abs(s1.std() / (math.sqrt(384) / math.sqrt(50)) - 1) < 0.05
print(f"mean {s1.mean():.2f}, spread {s1.std():.3f} vs theory {math.sqrt(384 / 50):.3f}")

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")